# Expfit notes 2: Jacobian, Hessian, and Levenberg-Marquardt

## Jacobian

Using the MSE rather than the RMSE means we have easy-to-compute jacobians and hessians.

First, we write out the MSE as $E$:
\begin{align}
   E &= \frac{1}{n} \sum \left(y_i - a - b e^{c x_i} \right)^2 \\
     &= \frac{1}{n} \sum y_i^2 + a^2 - 2 a y_i + 2 a b e^{c x_i} - 2 b y_i e^{c x_i} + b^2 e^{c x_i} \\
&\equiv \frac{1}{n} \sum y_i^2 + a^2 - 2 a y_i + 2 a b e_i - 2 b y_i e_i + b^2 e_i^2
\end{align}
where, for convenience, $e_i = e^{c x_i}$.

Then
\begin{align}
\frac{\partial E}{\partial a} &= \frac{1}{n} \sum 2 a - 2 y_i + 2 b e_i 
                              &&= \frac{2}{n} \sum a - y_i + b e_i \\
\frac{\partial E}{\partial b} &= \frac{1}{n} \sum 2 a e_i - 2 y_i e_i + 2 b e_i^2 
                              &&= \frac{2}{n} \sum (a - y_i + b e_i) e_i \\
\frac{\partial E}{\partial c} &= \frac{1}{n} \sum 2 a b x_i e_i - 2 b x_i y_i e_i + 2 b^2 x_i e_i^2
                              &&= \frac{2b}{n} \sum (a - y_i + b e_i) x_i e_i \\
\end{align}

This is good news, computationally.
In a compiled language, we could write
```
for i in ...
    e = exp(c * x[i])
    da_increment = a - y[i] + b * e
    db_increment = da_increment * e
    dc_increment = db_increment * x[i]
```

In Python/NumPy, it's faster to implement this with array operations
```
e = np.exp(c * x)                  # 2 array operations
f = a - v + b * e                  # 3
g = e * f                          # 1
da = 2 / n * np.sum(f)             # 1
db = 2 / n * np.sum(g)             # 1 
dc = 2 / n * np.sum(g * x) * b     # 2 
```
which works out as 10 array operations.

### Checking our workings

We can check against finite differences:

In [4]:
a, b, c = 5, 5, -3
n = 100
x = np.linspace(0, 1, n)
y = a + b * np.exp(c * x)

a0, b0, c0 = a + 1, b - 1, c + 1
y0 = f(x, a0, b0, c0)

def mse(x, y, a, b, c):
    return np.sum((y - a - b * np.exp(c * x))**2) / len(x)

def mse_jac(x, y, a, b, c):
    m = 1 / n
    e = np.exp(c * x)
    g = a - y + b * e
    eg = e * g
    da = 2 * m * np.sum(g)
    db = 2 * m * np.sum(eg)
    dc = 2 * m * np.sum(eg * x) * b
    return m * np.sum(g * g), da, db, dc

def mse_jac_fd(x, y, a, b, c, dp=1e-9):
    m = 1 / n
    r1 = mse(x, y, a, b, c)
    da = (mse(x, y, a + dp, b, c) - r1) / dp
    db = (mse(x, y, a, b + dp, c) - r1) / dp
    dc = (mse(x, y, a, b, c + dp) - r1) / dp
    return r1, da, db, dc

m0 = mse(x, y, a0, b0, c0)
m1, da1, db1, dc1 = mse_jac(x, y, a0, b0, c0)
m2, da2, db2, dc2 = mse_jac_fd(x, y, a0, b0, c0)

print(f'Plain mse {m0}')
print(f'  mse_jac {m1}')
print(f'  mse_fd  {m2}')
print(f'dE/da {da1}')
print(f'   fd {da2}')
print(f'dE/db {db1}')
print(f'   fd {db2}')
print(f'dE/dc {dc1}')
print(f'   fd {dc2}')

Plain mse 1.4223235997383419
  mse_jac 1.4223235997383419
  mse_fd  1.4223235997383419
dE/da 2.281170165188211
   fd 2.2811705857606057
dE/db 0.8347188315402757
   fd 0.8347191826629796
dE/dc 1.4618025059805984
   fd 1.461802900948328


And even run a plain gradient descent:

In [5]:
alpha = 0.7
a1, b1, c1 = a0, b0, c0
for i in range(1000):
    m1, da, db, dc = mse_jac(x, y, a1, b1, c1)
    a1 -= da * alpha
    b1 -= db * alpha
    c1 -= dc * alpha
        
print(i, m1)
print(f'  {a: } {a1: 6f}')
print(f'  {b: } {b1: 6f}')
print(f'  {c: } {c1: 6f}')

999 4.6560345431023555e-23
   5  5.000000
   5  5.000000
  -3 -3.000000


## Hessian

We can go a step further and find the Hessian

From $\frac{\partial E}{\partial a} = \frac{2}{n} \sum a - y_i + b e_i $
\begin{align}
\frac{\partial^2 E}{\partial a^2} 
    &= \frac{2}{n} \sum 1 = 2 \\
\frac{\partial^2 E}{\partial a \, \partial b} 
    &= \frac{2}{n} \sum e_i \\
\frac{\partial^2 E}{\partial a \, \partial c} 
    &= \frac{2b}{n} \sum x_i e_i \\
\end{align}

From $\frac{\partial E}{\partial b} = \frac{2}{n} \sum (a - y_i + b e_i) e_i $
\begin{align}
\frac{\partial^2 E}{\partial b \, \partial a} 
    &= \frac{2}{n} \sum e_i \\
\frac{\partial^2 E}{\partial b^2}
    &= \frac{2}{n} \sum e_i^2 \\
\frac{\partial^2 E}{\partial b \, \partial c}
    &= \frac{2}{n} \sum (a - y_i + 2 b e_i) x_i e_i
\end{align}

From $\frac{\partial E}{\partial c} = \frac{2b}{n} \sum (a - y_i + b e_i) x_i e_i $
\begin{align}
\frac{\partial^2 E}{\partial c \, \partial a}
    &= \frac{2b}{n} \sum x_i e_i \\
\frac{\partial^2 E}{\partial c \, \partial b}
    &= \frac{2}{n} \sum (a - y_i + 2 b e_i) x_i e_i   \\
\frac{\partial^2 E}{\partial c^2}
    &= \frac{2b}{n} \sum (a - y_i + 2 b e_i) x_i^2 e_i \\
\end{align}

**Sanity checks**: 
\begin{align}
\frac{\partial^2 E}{\partial a \, \partial b} &= \frac{\partial^2 E}{\partial b \, \partial a}, &&
\frac{\partial^2 E}{\partial a \, \partial c} &= \frac{\partial^2 E}{\partial c \, \partial a}, &&
\frac{\partial^2 E}{\partial b \, \partial c} &= \frac{\partial^2 E}{\partial c \, \partial b} 
\end{align}

### Checking our workings

In [6]:
def mse_jac_hes(x, y, p):
    a, b, c = p
    m = 1 / n
    e = np.exp(c * x)
    f = a - y + b * e
    ef = e * f
    mse = m * np.sum(f * f)

    # Jacobian
    jac = np.array([
        2 * m * np.sum(f),
        2 * m * np.sum(ef),
        2 * m * np.sum(ef * x) * b
    ])

    # Hessian
    ex = e * x
    aex = (a - y + 2 * b * e) * ex
    hes = np.array([
        [2, 2 * m * np.sum(e), 2 * b * m * np.sum(ex)],
        [0, 2 * m * np.sum(e * e), 2 * m * np.sum(aex)],
        [0, 0, 2 * m * b * np.sum(x * aex)],
    ])
    hes[1, 0] = hes[0, 1]
    hes[2, 0] = hes[0, 2]
    hes[2, 1] = hes[1, 2]
    
    return mse, jac, hes

m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, c0))

print(m1)
print(j1)
print(h1)

1.4223235997383419
[2.28117017 0.83471883 1.46180251]
[[2.         0.86740054 1.18144538]
 [0.86740054 0.49618302 0.81578101]
 [1.18144538 0.81578101 1.60402956]]


In [7]:
def mse_jac_hes_fd(x, y, p, dp=1e-6):
    a, b, c = p
    err = mse(x, y, a, b, c)
    da = lambda a, b, c: (mse(x, y, a + dp, b, c) - mse(x, y, a, b, c)) / dp
    db = lambda a, b, c: (mse(x, y, a, b + dp, c) - mse(x, y, a, b, c)) / dp
    dc = lambda a, b, c: (mse(x, y, a, b, c + dp) - mse(x, y, a, b, c)) / dp
    jac = np.array([da(a, b, c), db(a, b, c), dc(a, b, c)])
    hes = np.array([
        [(da(a + dp, b, c) - da(a, b, c)) / dp,
         (da(a, b + dp, c) - da(a, b, c)) / dp,
         (da(a, b, c + dp) - da(a, b, c)) / dp], 
        [(db(a + dp, b, c) - db(a, b, c)) / dp,
         (db(a, b + dp, c) - db(a, b, c)) / dp,
         (db(a, b, c + dp) - db(a, b, c)) / dp],
        [(dc(a + dp, b, c) - dc(a, b, c)) / dp,
         (dc(a, b + dp, c) - dc(a, b, c)) / dp,
         (dc(a, b, c + dp) - dc(a, b, c)) / dp]        
    ])
    
    return err, jac, hes

m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, c0))
m2, j2, h2 = mse_jac_hes_fd(x, y, (a0, b0, c0))
print(f'Plain mse {m0}')
print(f'  mse_jac {m1}')
print(f'  mse_fd  {m2}')
print()
print(f'dE/da {j1[0]}')
print(f'   fd {j2[0]}')
print(f'dE/db {j1[1]}')
print(f'   fd {j2[1]}')
print(f'dE/dc {j1[2]}')
print(f'   fd {j2[2]}')
print()
print(h1)
print()
print(h2)

Plain mse 1.4223235997383419
  mse_jac 1.4223235997383419
  mse_fd  1.4223235997383419

dE/da 2.281170165188211
   fd 2.281171165741114
dE/db 0.8347188315402757
   fd 0.8347190805224614
dE/dc 1.4618025059805984
   fd 1.461803308400178

[[2.         0.86740054 1.18144538]
 [0.86740054 0.49618302 0.81578101]
 [1.18144538 0.81578101 1.60402956]]

[[2.00039985 0.86708418 1.18083321]
 [0.86708418 0.49515947 0.8149037 ]
 [1.18083321 0.8149037  1.60316205]]


## Optimisation

Now that we have a Jacobian and Hessian, we can try classic optimisation methods.

### Newton's method

First, we try a plain Newton's method:
\begin{align}
p_{i+1} = p_i - H(p_i)^{-1} J(p_i)
\end{align}

In [8]:
ptrue = np.array([a, b, c])
p0 = np.array([a0, b0, c0], dtype=float)
m1, j1, h1 = mse_jac_hes(x, y, p0)
p1 = p0 - np.linalg.inv(h1) @ j1

print(f'True     {ptrue}')
print(f'Initial  {p0}')
print(f'One step {p1}')

True     [ 5  5 -3]
Initial  [ 6.  4. -2.]
One step [ 2.46233735 14.47653658 -5.63385256]


This isn't promising, but we can try a few more:

In [9]:
p1 = np.array([[a0, b0, c0]], dtype=float)
for i in range(10):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= 0.5 * np.linalg.inv(h1) @ j1
print(f'Solution: {p1}')
print(f'Error: {m1}')

Solution: [[ 4.99001201  5.00716354 -2.97847408]]
Error: 5.4459769596431235e-05


So with multiple steps we get very close, very quickly!

In [10]:
p1 = np.array([[a0, b0, c0]], dtype=float)
for i in range(20):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= np.linalg.inv(h1) @ j1
print(f'Solution: {p1}')
print(f'Error: {m1}')

Solution: [[-3.65388680e+07  9.79215203e+02  1.32210866e+01]]
Error: 2.8541850310169616e+16


But with more steps it gets worse.
We can fix this with a damping factor:

In [11]:
p1 = np.array([[a0, b0, c0]], dtype=float)
for i in range(20):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= 0.5 * np.linalg.inv(h1) @ j1
print(f'Solution: {p1}')
print(f'Error: {m1}')

Solution: [[ 4.99998992  5.00000706 -2.99997801]]
Error: 5.974891720861604e-11


### Comparison with Scipy

To get an idea of what "good performance" looks like, we can compare with SciPy, which uses BFGS: a method that _approximates_ the Hessian.

Unfortunately it expects seperate functions for error and jacobian, so we need to implement something like this:

In [12]:
class CachedHesJac:
    _p = None
    _mse = None
    _jac = None
    _hes = None
    def _do(self, p):
        if np.any(p != self._p):
            self._mse, self._jac, self._hes = mse_jac_hes(x, y, p)
            self._p = np.copy(p)
    def mse(self, p):
        self._do(p)
        return self._mse
    def jac(self, p):
        self._do(p)
        return self._jac
    def hes(self, p):
        self._do(p)
        return self._hes

which will avoid repeated calculations and make the comparison a bit fairer.

The [default method in scipy is BFGS](https://github.com/scipy/scipy/blob/8c75ae75176236f233824e9a0483c26a69e6dfec/scipy/optimize/_minimize.py#L606), and the [default tolerance is a gradient norm](https://github.com/scipy/scipy/blob/8c75ae75176236f233824e9a0483c26a69e6dfec/scipy/optimize/_minimize.py#L679) with [default value](https://github.com/scipy/scipy/blob/8c75ae75176236f233824e9a0483c26a69e6dfec/scipy/optimize/_optimize.py#L1346) 1e-5

In [13]:
import timeit
from scipy.optimize import minimize as fmin

chj = CachedHesJac()

time = timeit.default_timer()
r = fmin(chj.mse, p0, jac=chj.jac, method='bfgs', tol=1e-5)
time = timeit.default_timer() - time
print(r)
print(f'Time: {time}')

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 6.710597137344661e-10
        x: [ 5.000e+00  5.000e+00 -3.000e+00]
      nit: 14
      jac: [-7.402e-06 -6.258e-06  3.946e-06]
 hess_inv: [[ 5.844e+00 -3.500e+00 -9.726e+00]
            [-3.500e+00  7.492e+00  2.525e+00]
            [-9.726e+00  2.525e+00  2.041e+01]]
     nfev: 15
     njev: 15
Time: 0.003187880000041332


Without Jacobian:

In [14]:
time = timeit.default_timer()
r = fmin(chj.mse, p0, method='bfgs', tol=1e-5)
time = timeit.default_timer() - time
print(r)
print(f'Time: {time}')

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 6.716599359465852e-10
        x: [ 5.000e+00  5.000e+00 -3.000e+00]
      nit: 14
      jac: [-7.402e-06 -6.258e-06  3.946e-06]
 hess_inv: [[ 5.844e+00 -3.500e+00 -9.726e+00]
            [-3.500e+00  7.492e+00  2.525e+00]
            [-9.726e+00  2.525e+00  2.041e+01]]
     nfev: 60
     njev: 15
Time: 0.00818738499947358


Newton's method for the same number of iterations:

In [15]:
p1 = np.array([[a0, b0, c0]], dtype=float)
time = timeit.default_timer()
for i in range(14):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= 0.5 * np.linalg.inv(h1) @ j1
time = timeit.default_timer() - time
print(f'Solution: {p1}')
print(f'Error: {m1}')
print(f'Time: {time}')

Solution: [[ 4.99935649  5.00045141 -2.99859698]]
Error: 2.4244146915586865e-07
Time: 0.0012035099998684018


This is promising.

It may be faster to use "solve" instead of inverting and multiplying, but we can't tell with this benchmark:

In [16]:
p1 = np.array([[a0, b0, c0]], dtype=float)
time = timeit.default_timer()
for i in range(14):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= 0.5 * np.linalg.solve(h1, j1)
time = timeit.default_timer() - time
print(f'Solution: {p1}')
print(f'Error: {m1}')
print(f'Time: {time}')

Solution: [[ 4.99935649  5.00045141 -2.99859698]]
Error: 2.4244146915586865e-07
Time: 0.0011246280000705156


Finally, we can add in the gradient norm tolerance:

In [17]:
p1 = np.array([[a0, b0, c0]], dtype=float)
time = timeit.default_timer()
for i in range(1000):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= 0.5 * np.linalg.solve(h1, j1)
    if np.linalg.norm(j1) < 1e-5:
        break    
time = timeit.default_timer() - time
print(f'Solution: {p1}')
print(f'Error: {m1}')
print(f'Iterations: {i}')
print(f'Time: {time}')

Solution: [[ 4.99998992  5.00000706 -2.99997801]]
Error: 5.974891720861604e-11
Iterations: 19
Time: 0.0016914880006879685


### LM-style update

Levenberg:
\begin{align}
p_{i+1} = p_i - (H(p_i) + \lambda I)^{-1} J(p_i)
\end{align}

where usually $H$ is approximated by $J^\intercal J$, but here we'll use the true Hessian.

In [18]:
p1 = np.array([[a0, b0, c0]], dtype=float)
print(f'Initial {p1}')

time = timeit.default_timer()
alpha = 1
m1 = m_last = mse(x, y, *p1[0])
eye = np.eye(3)
for i in range(1000):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    if np.linalg.norm(j1) < 1e-5:
        break    
    ps = p1 - np.linalg.solve(h1 + alpha * eye, j1)
    ms = mse(x, y, *ps[0])
    if ms < m1:
        alpha *= 0.1
        p1 = ps
    elif ms > m1:
        alpha *= 10
time = timeit.default_timer() - time

print(f'Solution: {p1}')
print(f'Error: {m1}')
print(f'Iterations: {i}')
print(f'Time: {time}')

Initial [[ 6.  4. -2.]]
Solution: [[ 4.99999995  5.00000001 -2.99999987]]
Error: 4.0339001661945436e-16
Iterations: 6
Time: 0.0009465989996897406


This is _fast_.

Levenberg-Marquardt:
\begin{align}
p_{i+1} = p_i - (H(p_i) + \lambda \text{diag}(H))^{-1} J(p_i)
\end{align}

In [19]:
p1 = np.array([[a0, b0, c0]], dtype=float)
print(f'Initial {p1}')

time = timeit.default_timer()
alpha = 1
m1 = m_last = mse(x, y, *p1[0])
eye = np.eye(3)
for i in range(1000):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    if np.linalg.norm(j1) < 1e-5:
        break
    ps = p1 - np.linalg.solve(h1 + alpha * eye * h1, j1)
    ms = mse(x, y, *ps[0])
    if ms < m1:
        alpha *= 0.1
        p1 = ps
    elif ms > m1:
        alpha *= 10
time = timeit.default_timer() - time

print(f'Solution: {p1}')
print(f'Error: {m1}')
print(f'Iterations: {i}')
print(f'Time: {time}')

Initial [[ 6.  4. -2.]]
Solution: [[ 4.99998938  5.00000321 -2.99997659]]
Error: 1.1471355176432101e-11
Iterations: 5
Time: 0.0008908639993023826


Which shaves off another iteration for this test case.

### More tests

In [24]:
def opt(p0):
    p1 = np.array([p0], dtype=float)
    print(f'Initial {p1}')
    
    time = timeit.default_timer()
    alpha = 1
    m1 = m_last = mse(x, y, *p1[0])
    eye = np.eye(3)
    for i in range(1000):
        m1, j1, h1 = mse_jac_hes(x, y, p1[0])
        if np.linalg.norm(j1) < 1e-5:
            break
        ps = p1 - np.linalg.solve(h1 + alpha * eye * h1, j1)
        ms = mse(x, y, *ps[0])
        if ms < m1:
            alpha *= 0.1
            p1 = ps
        elif ms > m1:
            alpha *= 10
    time = timeit.default_timer() - time
    
    print(f'Solution: {p1}')
    print(f'Norm: {np.linalg.norm(j1)}')
    print(f'Error: {m1}')
    print(f'Iterations: {i}')
    print(f'Time: {time}')

opt([1, 1, 1])

Initial [[1. 1. 1.]]
Solution: [[ 5.  5. -3.]]
Norm: 2.0776568277107985e-10
Error: 1.2274222487374872e-19
Iterations: 9
Time: 0.00483358000019507


In [25]:
opt([10, 10, 10])

Initial [[10. 10. 10.]]
Solution: [[6.59421598e+00 1.48212311e-08 1.19527080e-09]]
Norm: 1.2614000352423026e-08
Error: 1.6998197929030272
Iterations: 27
Time: 0.006975156999033061


In [26]:
opt([-10, -10, -10])

Initial [[-10. -10. -10.]]
Solution: [[ 5.00000067  5.00000526 -3.00000578]]
Norm: 9.025003065411728e-07
Error: 2.945122411604627e-12
Iterations: 18
Time: 0.002144476000466966
